In [ ]:
#Downloading Dataset
!wget https://raw.githubusercontent.com/logpai/loghub/master/Linux/Linux_2k.log -O auth.log

In [ ]:
#Loading the logs
def load_logs(file_path):
    with open(file_path, "r") as file:
        logs = file.readlines()
    return logs

logs = load_logs("auth.log")
print("Total logs:", len(logs))

In [ ]:
#Filtering Suspicious Logs
def filter_logs(logs):
    keywords = [
        "Failed password",
        "invalid user",
        "Accepted password",
        "authentication failure",
        "sudo"
    ]
    return [log for log in logs if any(k in log for k in keywords)]

filtered_logs = filter_logs(logs)

print("Security related logs:", len(filtered_logs))

In [ ]:
#Showing suspicious entries
print("Sample suspicious entries:\n")

for log in filtered_logs[:5]:
    print(log.strip())

In [ ]:
#Install Free LLM
!pip install --upgrade transformers accelerate

In [ ]:
from transformers import pipeline
import torch

generator = pipeline(
    "text-generation",
    model="tiiuae/falcon-rw-1b",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device=0 if torch.cuda.is_available() else -1
)

print("Instruction model loaded successfully.")

In [ ]:
def prepare_logs_for_llm(logs, limit=10):
    selected = logs[:limit]
    formatted = "\n".join([log.strip() for log in selected])
    return formatted

In [ ]:
def analyze_logs(question, logs):

    failed_logs = [log for log in logs if "authentication failure" in log or "Failed password" in log]
    success_logs = [log for log in logs if "Accepted password" in log]

    failed_count = len(failed_logs)
    success_count = len(success_logs)

    question = question.lower()

    if "summarize" in question:
        return (
            f"The logs contain {failed_count} failed login attempts and "
            f"{success_count} successful logins. "
            "Repeated authentication failures suggest possible suspicious activity."
        )

    elif "brute" in question:
        if failed_count >= 5:
            return (
                f"There are {failed_count} failed login attempts. "
                "This volume of repeated failures may indicate a brute-force attack."
            )
        else:
            return (
                f"There are {failed_count} failed login attempts. "
                "This does not strongly indicate brute-force activity."
            )

    elif "repeated" in question:
        return (
            f"There are {failed_count} failed login attempts recorded in the logs, "
            "indicating repeated authentication failures."
        )

    else:
        return (
            f"There are {failed_count} failed login attempts and "
            f"{success_count} successful login events in the logs."
        )

In [ ]:
def security_copilot_interface(logs):

    print("     LLM Security Copilot System   ")
    print("Ask security-related questions.")
    print("Type 'exit' to stop.\n")

    while True:
        user_query = input("Your Question: ")

        if user_query.lower() == "exit":
            print("\nExiting Security Copilot.")
            break

        response = analyze_logs(user_query, logs)

        print("\nSecurity Copilot Response:\n")
        print(response)


In [ ]:
security_copilot_interface(filtered_logs)